# 11 — Explainable AI (XAI)
## Parkinson metabolic transcriptomic signature — GSE136666

This notebook explains the **pre-specified primary model from Notebook 10: Top-5 mechanistic genes + L2 Logistic Regression**.

For standardized feature \(z_j\):
\[
\mathrm{logit}\{P(PD)\}=\beta_0+\sum_j\beta_jz_j
\]
and the exact sample-specific gene contribution on the log-odds scale is:
\[
C_{ij}=\beta_jz_{ij}.
\]

**Scope:** exploratory model explanation only. With n=10 and upstream signature discovery involving GSE136666, these explanations are not independent validation and are not causal biological effects.

In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path(r"C:\Users\Camfe\pythonlab")
NB10_DIR = PROJECT_ROOT/"results"/"machine_learning"/"notebook10"
NB08_DIR = PROJECT_ROOT/"results"/"validation"/"notebook08"
NB09_DIR = PROJECT_ROOT/"results"/"networks"/"notebook09"
OUT_DIR = PROJECT_ROOT/"results"/"explainable_AI"/"notebook11"
FIG_DIR = PROJECT_ROOT/"figures"/"explainable_AI"/"notebook11"
OUT_DIR.mkdir(parents=True, exist_ok=True); FIG_DIR.mkdir(parents=True, exist_ok=True)
print("NB10:", NB10_DIR); print("OUT:", OUT_DIR)

## 1. Locate Notebook 10 outputs

In [ ]:
def locate(name, preferred):
    p=Path(preferred)
    if p.exists(): return p
    hits=list(PROJECT_ROOT.rglob(name))
    return hits[0] if hits else None

PRED_FILE=locate("primary_LOOCV_predictions.csv",NB10_DIR/"primary_LOOCV_predictions.csv")
COEF_FILE=locate("primary_logistic_coefficients.csv",NB10_DIR/"primary_logistic_coefficients.csv")
LOO_COEF_FILE=locate("primary_logistic_LOOCV_coefficients.csv",NB10_DIR/"primary_logistic_LOOCV_coefficients.csv")
STABILITY_FILE=locate("primary_logistic_coefficient_stability.csv",NB10_DIR/"primary_logistic_coefficient_stability.csv")
XAI_INPUT_FILE=locate("Notebook11_XAI_feature_input.csv",NB10_DIR/"Notebook11_XAI_feature_input.csv")
for k,v in {"pred":PRED_FILE,"coef":COEF_FILE,"loo_coef":LOO_COEF_FILE,"stability":STABILITY_FILE,"xai_input":XAI_INPUT_FILE}.items():
    print(k,":",v)
if any(v is None for v in [PRED_FILE,COEF_FILE,LOO_COEF_FILE,STABILITY_FILE,XAI_INPUT_FILE]):
    raise FileNotFoundError("One or more required Notebook 10 XAI outputs are missing.")
print("✓ Essential Notebook 10 inputs found.")

## 2. Load and audit inputs

In [ ]:
pred=pd.read_csv(PRED_FILE); coef=pd.read_csv(COEF_FILE)
loo_coef=pd.read_csv(LOO_COEF_FILE); stability=pd.read_csv(STABILITY_FILE)
xai_input=pd.read_csv(XAI_INPUT_FILE)
for name,df in [("pred",pred),("coef",coef),("loo_coef",loo_coef),("stability",stability),("xai_input",xai_input)]:
    print(name,df.shape,df.columns.tolist())
display(coef.head()); display(xai_input.head())

## 3. Detect columns and reconstruct Top-5 matrix

In [ ]:
def detect(df,names=(),contains=(),required=True,label="column"):
    low={str(c).lower():c for c in df.columns}
    for n in names:
        if n.lower() in low: return low[n.lower()]
    for c in df.columns:
        cl=str(c).lower()
        if contains and all(t.lower() in cl for t in contains): return c
    if required: raise KeyError(f"Could not detect {label}. Columns: {df.columns.tolist()}")
    return None

CG=detect(coef,["gene","signature_gene","feature","Symbol"],required=True,label="coefficient gene")
CV=detect(coef,["coefficient","coef","beta"],contains=["coef"],required=True,label="coefficient")
primary_genes=coef[CG].astype(str).tolist()

SC=detect(xai_input,["sample","sample_id"],contains=["sample"],required=True,label="sample")
YC=detect(xai_input,["y","label","target","disease_binary"],required=False)
if YC is None:
    DC=detect(xai_input,["disease","group","condition"],required=True,label="disease")
    ys=xai_input[DC].astype(str).str.upper().map({"CONTROL":0,"CTRL":0,"C":0,"PD":1,"PARKINSON":1})
else:
    ys=pd.to_numeric(xai_input[YC],errors="coerce")
if ys.isna().any(): raise RuntimeError("Could not construct Control=0 / PD=1 labels.")
missing=[g for g in primary_genes if g not in xai_input.columns]
if missing: raise KeyError(f"Primary genes missing from XAI input: {missing}")

X=xai_input[primary_genes].apply(pd.to_numeric,errors="raise").copy()
X.index=xai_input[SC].astype(str); y=ys.astype(int).to_numpy()
if X.shape[0]!=10 or np.sum(y==0)!=5 or np.sum(y==1)!=5:
    raise RuntimeError(f"Expected n=10, 5 Control/5 PD; got {X.shape}, {pd.Series(y).value_counts().to_dict()}")
print("Primary genes:",primary_genes); print("X:",X.shape)
display(X)

## 4. Refit primary model for explanation
Notebook 10 remains the performance analysis. This all-sample refit is used only to explain the final fitted linear model.

In [ ]:
scaler=StandardScaler(); Z=scaler.fit_transform(X)
model=LogisticRegression(penalty="l2",solver="liblinear",random_state=42,max_iter=5000)
model.fit(Z,y)
beta=pd.Series(model.coef_.ravel(),index=primary_genes,name="beta")
intercept=float(model.intercept_[0])

export_beta=(coef.assign(_gene=coef[CG].astype(str)).set_index("_gene")[CV].reindex(primary_genes).astype(float))
coef_compare=pd.DataFrame({"gene":primary_genes,"Notebook10_exported_beta":export_beta.values,"Notebook11_refit_beta":beta.values})
coef_compare["absolute_difference"]=(coef_compare["Notebook10_exported_beta"]-coef_compare["Notebook11_refit_beta"]).abs()
display(coef_compare); print("Intercept:",intercept)

## 5. Exact sample-specific log-odds contributions

In [ ]:
contrib=pd.DataFrame(Z*beta.to_numpy(),index=X.index,columns=primary_genes)
logodds=intercept+contrib.sum(axis=1)
prob=1/(1+np.exp(-logodds))
if not np.allclose(model.decision_function(Z),logodds.to_numpy(),atol=1e-10):
    raise RuntimeError("Contribution decomposition failed.")
sample_summary=pd.DataFrame({
    "sample":X.index,
    "true_class":np.where(y==1,"PD","Control"),
    "fitted_probability_PD":prob,
    "fitted_class":np.where(prob>=.5,"PD","Control"),
    "intercept_log_odds":intercept,
    "gene_contribution_sum":contrib.sum(axis=1).values,
    "total_log_odds":logodds.values
})
print("✓ Exact log-odds decomposition verified.")
display(sample_summary); display(contrib)

## 6. Global coefficient interpretation

In [ ]:
global_xai=pd.DataFrame({
    "gene":primary_genes,
    "standardized_beta":beta.values,
    "absolute_beta":np.abs(beta.values),
    "model_direction":np.where(beta.values>0,"toward_PD","toward_Control"),
    "mean_absolute_sample_contribution":contrib.abs().mean(axis=0).reindex(primary_genes).values
})
global_xai["absolute_beta_rank"]=global_xai["absolute_beta"].rank(ascending=False,method="min").astype(int)
global_xai=global_xai.sort_values("absolute_beta",ascending=False).reset_index(drop=True)
display(global_xai)

fig,ax=plt.subplots(figsize=(8,4.8))
p=global_xai.sort_values("standardized_beta")
ax.barh(p["gene"],p["standardized_beta"]); ax.axvline(0,linewidth=1)
ax.set_xlabel("Standardized logistic coefficient (β)"); ax.set_title("Primary model coefficients")
fig.tight_layout(); fig.savefig(FIG_DIR/"primary_standardized_coefficients.png",dpi=300,bbox_inches="tight"); plt.show()

## 7. LOOCV coefficient sign stability

In [ ]:
LG=detect(loo_coef,["gene","signature_gene","feature","Symbol"],required=True,label="LOOCV gene")
LV=detect(loo_coef,["coefficient","coef","beta"],contains=["coef"],required=True,label="LOOCV coefficient")
lw=loo_coef[[LG,LV]].copy(); lw.columns=["gene","coefficient"]
lw["gene"]=lw["gene"].astype(str); lw["coefficient"]=pd.to_numeric(lw["coefficient"],errors="coerce"); lw=lw.dropna()
stab=lw.groupby("gene")["coefficient"].agg(loo_mean_beta="mean",loo_sd_beta="std",loo_min_beta="min",loo_max_beta="max",n_folds="count").reset_index()
sf=(lw.assign(pos=lw.coefficient>0,neg=lw.coefficient<0).groupby("gene").agg(frac_positive=("pos","mean"),frac_negative=("neg","mean")).reset_index())
stab=stab.merge(sf,on="gene"); stab["dominant_sign_fraction"]=stab[["frac_positive","frac_negative"]].max(axis=1)
stab["sign_stable_all_folds"]=stab["dominant_sign_fraction"].eq(1.0)
display(stab)

fig,ax=plt.subplots(figsize=(8,4.8))
t=stab.set_index("gene").reindex(primary_genes).dropna()
xx=np.arange(len(t)); ax.errorbar(xx,t.loo_mean_beta,yerr=t.loo_sd_beta.fillna(0),fmt="o",capsize=4); ax.axhline(0,linewidth=1)
ax.set_xticks(xx); ax.set_xticklabels(t.index,rotation=45,ha="right"); ax.set_ylabel("LOOCV coefficient: mean ± SD")
ax.set_title("Coefficient stability across LOOCV folds")
fig.tight_layout(); fig.savefig(FIG_DIR/"LOOCV_coefficient_stability.png",dpi=300,bbox_inches="tight"); plt.show()

## 8. Sample × gene contribution heatmap

In [ ]:
fig,ax=plt.subplots(figsize=(9,6))
im=ax.imshow(contrib.to_numpy(),aspect="auto")
ax.set_xticks(np.arange(len(primary_genes))); ax.set_xticklabels(primary_genes,rotation=45,ha="right")
ax.set_yticks(np.arange(len(contrib))); ax.set_yticklabels([f"{s} ({'PD' if yy else 'Control'})" for s,yy in zip(contrib.index,y)])
ax.set_title("Exact gene contributions to fitted log-odds"); ax.set_xlabel("Gene"); ax.set_ylabel("Sample")
fig.colorbar(im,ax=ax,label="β × standardized expression")
fig.tight_layout(); fig.savefig(FIG_DIR/"sample_gene_logodds_contributions.png",dpi=300,bbox_inches="tight"); plt.show()

## 9. Long-form contributions and dominant sample driver

In [ ]:
long_contrib=contrib.reset_index(names="sample").melt(id_vars="sample",var_name="gene",value_name="log_odds_contribution")
long_contrib["absolute_contribution"]=long_contrib.log_odds_contribution.abs()
long_contrib["direction"]=np.where(long_contrib.log_odds_contribution>0,"toward_PD",np.where(long_contrib.log_odds_contribution<0,"toward_Control","neutral"))
dominant=(long_contrib.sort_values(["sample","absolute_contribution"],ascending=[True,False]).groupby("sample",as_index=False).head(1)
          .rename(columns={"gene":"dominant_gene","log_odds_contribution":"dominant_log_odds_contribution","direction":"dominant_direction"}))
sample_summary=sample_summary.merge(dominant[["sample","dominant_gene","dominant_log_odds_contribution","dominant_direction"]],on="sample",how="left")
display(sample_summary)

## 10. Integrate cross-dataset and mechanistic evidence (interpretation only)

In [ ]:
def maybe(name,preferred):
    p=Path(preferred)
    if p.exists(): return p
    h=list(PROJECT_ROOT.rglob(name)); return h[0] if h else None
def gcol(df): return detect(df,["signature_gene","gene","Symbol","symbol"],required=False)

EV=maybe("cross_dataset_evidence_table.csv",NB08_DIR/"cross_dataset_evidence_table.csv")
ME=maybe("mechanistic_gene_priority.csv",NB09_DIR/"mechanistic_gene_priority.csv")
integrated=global_xai.copy()
if EV:
    d=pd.read_csv(EV); gc=gcol(d)
    if gc:
        d=d.rename(columns={gc:"gene"}).drop_duplicates("gene")
        keep=["gene"]+[c for c in ["cross_dataset_score","final_validation_class","concordance_fraction","n_sn_FDR","n_both_FDR"] if c in d.columns]
        integrated=integrated.merge(d[keep],on="gene",how="left")
        print("✓ Notebook 08 evidence joined.")
if ME:
    d=pd.read_csv(ME); gc=gcol(d)
    if gc:
        d=d.rename(columns={gc:"gene"}).drop_duplicates("gene")
        keep=["gene"]+[c for c in ["mechanistic_priority_score","mechanistic_rank","degree","betweenness","theme_count"] if c in d.columns and c not in integrated.columns]
        integrated=integrated.merge(d[keep],on="gene",how="left")
        print("✓ Notebook 09 evidence joined.")
display(integrated)

## 11. Transparent XAI instead of unnecessary SHAP dependency
For standardized linear logistic regression, \(eta_jz_{ij}\) is already the exact additive decomposition of the model log-odds around its intercept. No external SHAP package is required. A positive contribution pushes this fitted model toward PD; a negative contribution pushes it toward Control.

In [ ]:
print("Exact contribution matrix available as `contrib`.")
print("No external SHAP dependency required.")

## 12. Export outputs

In [ ]:
global_xai.to_csv(OUT_DIR/"primary_global_XAI_coefficients.csv",index=False)
stab.to_csv(OUT_DIR/"primary_LOOCV_coefficient_stability_recomputed.csv",index=False)
contrib.to_csv(OUT_DIR/"sample_gene_logodds_contribution_matrix.csv")
long_contrib.to_csv(OUT_DIR/"sample_gene_logodds_contributions_long.csv",index=False)
sample_summary.to_csv(OUT_DIR/"sample_level_XAI_summary.csv",index=False)
integrated.to_csv(OUT_DIR/"integrated_gene_XAI_evidence.csv",index=False)
coef_compare.to_csv(OUT_DIR/"Notebook10_vs_Notebook11_coefficients.csv",index=False)

interpretation=integrated.copy()
interpretation["XAI_interpretation"]=np.where(
    interpretation.standardized_beta>0,
    "Higher standardized expression pushes the fitted model toward PD",
    "Higher standardized expression pushes the fitted model toward Control"
)
interpretation=interpretation.merge(stab[["gene","dominant_sign_fraction","sign_stable_all_folds"]],on="gene",how="left")
interpretation.to_csv(OUT_DIR/"thesis_XAI_interpretation_table.csv",index=False)
display(interpretation)

## 13. Thesis-ready interpretation

> Explainability analysis was performed on the pre-specified Top-5 L2 logistic regression model. Because predictors were standardized, each coefficient represents the change in fitted log-odds associated with a one-standard-deviation increase in the corresponding transcript while holding the remaining model features constant. Sample-specific contributions were calculated exactly as the product of the standardized feature value and its fitted coefficient. Coefficient direction and stability were additionally examined across leave-one-out folds. These analyses characterize the behavior of the fitted exploratory classifier and should not be interpreted as causal effects, independent biomarker validation, protein-level regulation, enzyme activity, or metabolic flux.

Because the same small bulk cohort contributed to upstream signature discovery, XAI results remain exploratory/apparent explanations of the fitted model.

## 14. Final checkpoints

In [ ]:
checks={
 "n_samples_is_10":X.shape[0]==10,
 "five_control_five_PD":np.sum(y==0)==5 and np.sum(y==1)==5,
 "at_least_3_features":X.shape[1]>=3,
 "finite_X":np.isfinite(X.to_numpy()).all(),
 "finite_contributions":np.isfinite(contrib.to_numpy()).all(),
 "exact_logodds_decomposition":np.allclose(model.decision_function(Z),logodds.to_numpy(),atol=1e-10),
}
check_df=pd.DataFrame({"check":list(checks),"passed":list(checks.values())}); display(check_df)
if not all(checks.values()):
    raise RuntimeError("Notebook 11 failed checks: "+", ".join(k for k,v in checks.items() if not v))

manifest={
 "notebook":"11_explainable_AI",
 "analysis_type":"exploratory model explanation",
 "primary_model":"Top-5 mechanistic genes + L2 Logistic Regression",
 "n_samples":int(X.shape[0]),"n_features":int(X.shape[1]),"features":list(X.columns),
 "explanation_scale":"log-odds",
 "contribution_definition":"standardized_feature * logistic_coefficient",
 "causal_interpretation":False,"independent_validation":False,
 "output_directory":str(OUT_DIR),"figure_directory":str(FIG_DIR)
}
with open(OUT_DIR/"notebook11_manifest.json","w",encoding="utf-8") as f: json.dump(manifest,f,indent=2,ensure_ascii=False)
check_df.to_csv(OUT_DIR/"notebook11_final_checkpoint.csv",index=False)

print("\n✓ All Notebook 11 checks passed.")
print("Notebook 11 completed successfully.")
print("\nNext:\n12_integrated_results.ipynb")